# Can an application redirect a voice shopping assistant after its tent explanation begins?

## 1. Before You Begin

We are building a Contoso Outdoors voice assistant that begins explaining the
TrailMaster X4 Tent, then receives a trusted application redirect to discuss
the Adventurer Pro Backpack. Our single focus is **full-duplex conversation
redirection**.

Deploy `gpt-live-1` in Microsoft Foundry, run `az login`, and configure the
variables checked in section 2. The default path uses no microphone: it streams
a committed shopper question as PCM audio, and received audio is saved under
`output/`. GPT-Live
voice duration is billed separately from delegated backend work; see
[current pricing](https://azure.microsoft.com/pricing/details/azure-openai/).


In [ ]:
# 2. Verify your environment
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

def find_assets() -> Path:
    """Find the shared data whether Jupyter starts here or at the repo root."""
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / "models/azure-openai/shared/contoso-outdoors"
        if (candidate / "products.json").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find models/azure-openai/shared/contoso-outdoors. "
        "Run this notebook from a checkout of the model-releases repository."
    )


required = [
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_GPT_LIVE_1_DEPLOYMENT",
]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise EnvironmentError(
        f"Missing {missing}. Copy scripts/sample.env to .env, add the values, "
        "and review models/quickstart/README.md."
    )

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
deployment = os.environ["AZURE_OPENAI_GPT_LIVE_1_DEPLOYMENT"]
assets = find_assets()
output_dir = assets.parents[1] / "gpt-live-1" / "output"
output_dir.mkdir(exist_ok=True)

for path in (
    assets / "products.json",
    assets / "manuals/product_info_1.md",
    assets / "manuals/product_info_2.md",
    assets / "audio/shopper-tent-question.wav",
):
    if not path.is_file():
        raise FileNotFoundError(path)

print(f"Environment ready for deployment: {deployment}")
print(f"Generated audio will be written to: {output_dir.resolve()}")


## 3. Prepare grounded product commentary

The application owns the product facts. We take short statements directly from
the committed catalog and manuals, then keep the explanation and redirect
separate so the event sequence is easy to inspect.


In [ ]:
# 4. Build the initial explanation and redirect from local assets
import json
from urllib.parse import urlparse

products = {
    product["id"]: product
    for product in json.loads((assets / "products.json").read_text(encoding="utf-8"))
}
tent_manual = (assets / "manuals/product_info_1.md").read_text(encoding="utf-8")
backpack_manual = (assets / "manuals/product_info_2.md").read_text(encoding="utf-8")

assert products[1]["name"] == "TrailMaster X4 Tent"
assert products[2]["name"] == "Adventurer Pro Backpack"
assert "3-season" in tent_manual
assert "40 liters" in backpack_manual

tent_context = (
    "Product 1 is the TrailMaster X4 Tent, a four-person, three-season tent "
    "with a rainfly. When asked about it, begin with setup guidance."
)
redirect_instruction = (
    "Stop the tent explanation at the next natural boundary. The shopper has "
    "redirected the conversation to product 2. Discuss only the backpack now."
)
backpack_commentary = (
    "Explain that product 2, the Adventurer Pro Backpack, has a 40-liter "
    "capacity and hydration compatibility. Mention that the technical specs "
    "say a rain cover is not included."
)
shopper_audio = assets / "audio/shopper-tent-question.wav"

parsed = urlparse(endpoint)
if parsed.scheme not in {"http", "https"} or not parsed.netloc:
    raise ValueError("AZURE_OPENAI_ENDPOINT must be an HTTP(S) resource endpoint")
live_url = f"wss://{parsed.netloc}/openai/v1/live/sessions"
print(live_url)


## 5. Redirect after spoken output begins

The client streams a committed PCM shopper question. On the first
`session.output_audio.delta`, it appends a trusted redirect instruction and
new commentary. This demonstrates an application-driven redirect without
microphone hardware. Transcript fragments can interleave and are not
authoritative turn boundaries.


In [ ]:
# 6. Run the bounded WebSocket session and save returned audio
import asyncio
import base64
import inspect
import wave

from azure.identity import DefaultAzureCredential
from websockets.asyncio.client import connect


async def run_redirect_session():
    events = []
    transcript = []
    audio_chunks = []
    redirect_sent = False
    redirect_acks = set()
    redirect_audio_start_ms = None
    redirected_transcript = []

    # websockets renamed this option in version 14; support both installed APIs.
    header_name = (
        "additional_headers"
        if "additional_headers" in inspect.signature(connect).parameters
        else "extra_headers"
    )
    # For local development, DefaultAzureCredential reuses the active Azure CLI login.
    access_token = DefaultAzureCredential().get_token(
        "https://cognitiveservices.azure.com/.default"
    ).token
    connect_options = {
        header_name: {"Authorization": f"Bearer {access_token}"}
    }

    async with connect(live_url, **connect_options) as websocket:
        await websocket.send(
            json.dumps(
                {
                    "type": "session.start",
                    "event_id": "start_contoso_session",
                    "session": {
                        "model": deployment,
                        "instructions": (
                            "Speak concisely. Use only facts supplied by the "
                            "application. Adapt immediately when redirected. "
                            f"{tent_context}"
                        ),
                        "audio": {"output": {"voice": "marin"}},
                        "delegation": {"type": "client"},
                    },
                }
            )
        )

        started = json.loads(await asyncio.wait_for(websocket.recv(), timeout=20))
        if started.get("type") == "error":
            raise RuntimeError(started)
        if started.get("type") != "session.started":
            raise RuntimeError(f"Expected session.started, received {started}")
        events.append(started)

        # Stream the committed PCM16 fixture in 100 ms chunks, then one second
        # of silence so the service can detect the end of the shopper's turn.
        with wave.open(str(shopper_audio), "rb") as wav_file:
            if (
                wav_file.getnchannels() != 1
                or wav_file.getsampwidth() != 2
                or wav_file.getframerate() != 24_000
            ):
                raise ValueError("Shopper fixture must be mono PCM16 at 24 kHz")
            input_pcm = wav_file.readframes(wav_file.getnframes())
        input_pcm += b"\x00" * 48_000
        chunk_bytes = 4_800
        for index, offset in enumerate(range(0, len(input_pcm), chunk_bytes)):
            chunk = input_pcm[offset : offset + chunk_bytes]
            await websocket.send(
                json.dumps(
                    {
                        "type": "session.input_audio.append",
                        "event_id": f"shopper_audio_{index}",
                        "audio": base64.b64encode(chunk).decode("ascii"),
                    }
                )
            )
            await asyncio.sleep(0.1)

        # Stop only after both redirect appends are acknowledged, the transcript
        # names the backpack, and post-ack audio advances three seconds. GPT-Live
        # doesn't emit an output-audio-done event.
        deadline = asyncio.get_running_loop().time() + 45
        while asyncio.get_running_loop().time() < deadline:
            try:
                event = json.loads(await asyncio.wait_for(websocket.recv(), timeout=5))
            except asyncio.TimeoutError:
                continue
            events.append(event)
            event_type = event.get("type")
            if event_type == "error":
                raise RuntimeError(event)
            if redirect_sent and event_type in {
                "session.instructions.appended",
                "session.commentary.appended",
            }:
                redirect_acks.add(event_type)
            if event_type in {
                "session.input_transcript.delta",
                "session.output_transcript.delta",
            }:
                fragment = {
                    "speaker": "assistant" if "output" in event_type else "shopper",
                    "text": event.get("delta", ""),
                    "start_ms": event.get("start_ms"),
                    "end_ms": event.get("end_ms"),
                }
                transcript.append(fragment)
                if (
                    event_type == "session.output_transcript.delta"
                    and len(redirect_acks) == 2
                ):
                    redirected_transcript.append(fragment["text"])
            if event_type == "session.output_audio.delta":
                audio_chunks.append(base64.b64decode(event["delta"]))
                if not redirect_sent:
                    redirect_sent = True
                    await websocket.send(
                        json.dumps(
                            {
                                "type": "session.instructions.append",
                                "event_id": "redirect_to_backpack",
                                "delegation_id": None,
                                "content": redirect_instruction,
                            }
                        )
                    )
                    await websocket.send(
                        json.dumps(
                            {
                                "type": "session.commentary.append",
                                "event_id": "explain_backpack",
                                "delegation_id": None,
                                "content": backpack_commentary,
                            }
                        )
                    )
                elif len(redirect_acks) == 2:
                    if redirect_audio_start_ms is None:
                        redirect_audio_start_ms = event.get("start_ms", 0)
                    redirected_text = "".join(redirected_transcript).lower()
                    names_backpack = any(
                        term in redirected_text for term in ("backpack", "adventurer pro")
                    )
                    if (
                        names_backpack
                        and event.get("end_ms", 0) >= redirect_audio_start_ms + 3000
                    ):
                        break

        await websocket.send(
            json.dumps({"type": "session.close", "event_id": "close_contoso_session"})
        )
        final_usage = None
        while True:
            event = json.loads(await asyncio.wait_for(websocket.recv(), timeout=20))
            events.append(event)
            if event.get("type") == "error":
                raise RuntimeError(event)
            if event.get("type") == "session.closed":
                final_usage = event.get("usage")
                break

    redirected_text = "".join(redirected_transcript).lower()
    if len(redirect_acks) != 2 or not audio_chunks:
        raise RuntimeError("The session ended before both redirect appends were acknowledged")
    if not any(term in redirected_text for term in ("backpack", "adventurer pro")):
        raise RuntimeError(
            "The session ended without transcript evidence of the backpack redirect"
        )

    audio_path = output_dir / "gpt-live-1-redirect.wav"
    with wave.open(str(audio_path), "wb") as wav_file:
        wav_file.setnchannels(1)
        wav_file.setsampwidth(2)
        wav_file.setframerate(24_000)
        wav_file.writeframes(b"".join(audio_chunks))

    transcript_path = output_dir / "gpt-live-1-transcript.json"
    transcript_path.write_text(json.dumps(transcript, indent=2), encoding="utf-8")
    return audio_path, transcript_path, final_usage, events


audio_path, transcript_path, final_usage, events = await run_redirect_session()
print(f"Saved audio: {audio_path}")
print(f"Saved transcript fragments: {transcript_path}")
print(f"Final cumulative usage: {final_usage}")
print("Event sequence:", [event.get("type") for event in events])


## 7. Your Turn to Explore

- Redirect from setup instructions to tent care instead of changing products.
- Compare a shorter and longer delay before the redirect.
- Replace application commentary with real PCM input in a separate,
  microphone-enabled client.


## 8. Summary

We used GPT-Live-1 for one capability: redirecting a spoken explanation while
the session remained active. The application supplied trusted facts, observed
the first audio delta, redirected the conversation, and preserved the returned
audio and transcript fragments. Review the
[audio and speech primer](../../../docs/primers/audio-speech.md) before choosing
between GPT-Live, Realtime, and a chained speech pipeline.


## 9. References

- [GPT-Live-1 model card](https://ai.azure.com/catalog/models/gpt-live-1) — Foundry catalog entry.
- [What is GPT-Live?](https://learn.microsoft.com/en-us/azure/foundry/openai/concepts/gpt-live) — full-duplex behavior and interruption semantics.
- [Use GPT-Live for real-time voice](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/gpt-live) — WebSocket lifecycle and PCM format.
- [GPT-Live event API reference](https://learn.microsoft.com/en-us/azure/foundry/openai/gpt-live-reference) — event fields and acknowledgments.
